# Student Performance Analysis and Grade Prediction

**BharatiCares Certificate Verification Project Submission**

This notebook performs exploratory data analysis (EDA) on the UCI Student Performance dataset and trains a reproducible regression model to estimate final mathematics grade (`G3`). The model is educational and must not be used to make high-stakes decisions about students.

**Reproducibility:** random seed `42`; fixed test split; metrics are calculated when you run the notebook. No performance metrics are invented in advance.

## 1. Project objectives

- Load a real, publicly documented student-performance dataset.
- Inspect the data, missing values, duplicates, distributions, and relationships.
- Visualise selected factors associated with final grade.
- Compare a simple baseline with a Random Forest regression model.
- Report MAE, RMSE, and R² on a held-out test set.

**Dataset source:** UCI Machine Learning Repository, *Student Performance* (ID 320): https://archive.ics.uci.edu/dataset/320/student+performance

This notebook uses the mathematics file (`student-mat.csv`, 395 records).

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn requests

In [ ]:
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 2. Load the real dataset

The notebook first downloads the official UCI ZIP archive. If the download fails, it tries a public mirror of the same UCI file. If both sources are unavailable, download `student-mat.csv` from the UCI dataset page and place it next to this notebook.

In [ ]:
def load_student_math_data():
    official_url = 'https://archive.ics.uci.edu/static/public/320/student+performance.zip'
    try:
        response = requests.get(official_url, timeout=30)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
            with archive.open('student-mat.csv') as file:
                return pd.read_csv(file, sep=';'), 'UCI official ZIP archive'
    except Exception as official_error:
        # Public mirror fallback; the README identifies the canonical UCI source.
        mirror_url = 'https://raw.githubusercontent.com/justmarkham/DAT8/master/data/student-mat.csv'
        try:
            response = requests.get(mirror_url, timeout=30)
            response.raise_for_status()
            return pd.read_csv(io.BytesIO(response.content), sep=';'), 'public mirror (UCI dataset copy)'
        except Exception as mirror_error:
            try:
                local = pd.read_csv('student-mat.csv', sep=';')
                return local, 'local student-mat.csv file'
            except Exception:
                raise RuntimeError(
                    'Could not download the dataset. Download student-mat.csv from '
                    'https://archive.ics.uci.edu/dataset/320/student+performance and place it beside this notebook.'
                ) from mirror_error

data, data_source = load_student_math_data()
print('Loaded from:', data_source)
print('Dataset shape:', data.shape)
display(data.head())

## 3. Data quality and overview

The dataset uses semicolon-separated values. The target variable is `G3`, the final mathematics grade on a 0–20 scale. `G1` and `G2` are earlier-period grades and are included in the main model; this means the model is not a purely early-term prediction model.

In [ ]:
print('Rows, columns:', data.shape)
print('\nColumn types:')
print(data.dtypes.value_counts())
print('\nMissing values (top 15):')
print(data.isna().sum().sort_values(ascending=False).head(15))
print('\nExact duplicate rows:', data.duplicated().sum())
print('\nNumeric summary:')
display(data.describe().T)

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data['G3'], bins=21, kde=True)
plt.title('Distribution of final mathematics grades (G3)')
plt.xlabel('Final grade (0–20)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(data=data, x='G1', y='G3', alpha=0.7)
plt.title('Earlier-period grade (G1) vs final grade (G3)')
plt.tight_layout()
plt.show()

## 4. Explore relationships

Correlation is descriptive, not proof of causation. The correlation heatmap below includes numeric columns only.

In [ ]:
numeric_data = data.select_dtypes(include=np.number)
correlations = numeric_data.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(correlations, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap of numeric variables')
plt.tight_layout()
plt.show()

if 'studytime' in data.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=data, x='studytime', y='G3')
    plt.title('Final grade by reported weekly study time')
    plt.xlabel('Study time category (dataset coding)')
    plt.tight_layout()
    plt.show()

## 5. Prepare features and train models

Categorical columns are one-hot encoded. Numeric columns are imputed with their median. The held-out test set is 20% of the data, using seed 42. A mean-prediction baseline is compared with a Random Forest model.

In [ ]:
target = 'G3'
X = data.drop(columns=[target]).copy()
y = data[target].copy()

categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, min_samples_leaf=2
    ))
])
model.fit(X_train, y_train)

pred_baseline = baseline.predict(X_test)
pred_model = model.predict(X_test)

def regression_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted)
    }

results = pd.DataFrame({
    'Mean baseline': regression_metrics(y_test, pred_baseline),
    'Random Forest': regression_metrics(y_test, pred_model)
}).T
print('Held-out test metrics (computed on this run):')
display(results.round(3))

## 6. Inspect predictions

Use these examples as a demonstration, not as authoritative judgments about individual students. Predictions can be wrong, and the dataset is relatively small and context-specific.

In [ ]:
prediction_check = pd.DataFrame({
    'Actual G3': y_test.reset_index(drop=True),
    'Predicted G3': np.round(pred_model, 1)
})
prediction_check['Absolute error'] = (prediction_check['Actual G3'] - prediction_check['Predicted G3']).abs()
display(prediction_check.head(10))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_model, alpha=0.7)
plt.plot([0, 20], [0, 20], linestyle='--')
plt.xlim(0, 20)
plt.ylim(0, 20)
plt.xlabel('Actual G3')
plt.ylabel('Predicted G3')
plt.title('Actual vs predicted final grade')
plt.tight_layout()
plt.show()

## 7. Findings and limitations

After running all cells, use the actual tables and plots above to write findings. Do not claim a feature causes grade changes merely because it is correlated with `G3`.

**Limitations**
- The data represents two Portuguese schools and may not generalise to other regions or education systems.
- `G1` and `G2` are strong prior-grade signals, so results may overstate performance for early intervention scenarios.
- A single train/test split gives one estimate; cross-validation and external validation would be needed for stronger evidence.
- Small groups and sensitive attributes require care; predictions should not be used to penalise students.

## 8. Conclusion

This project demonstrates a complete beginner-friendly data workflow: loading a real dataset, checking data quality, exploratory visualisation, preprocessing, baseline comparison, and regression evaluation. The printed metrics are reproducible for the specified dataset and software environment, subject to library-version differences.